In [ ]:
import pandas as pd
import re
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load dataset
df = pd.read_csv("train.csv", engine="python", on_bad_lines="skip")

# Combine toxicity labels
df['toxic_label'] = df[
    ['toxic', 'severe_toxic', 'obscene',
     'threat', 'insult', 'identity_hate']
].max(axis=1)

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['comment_text'] = df['comment_text'].apply(clean_text)

X = df['comment_text']
y = df['toxic_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Tokenization
MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

# Model
model = Sequential()
model.add(Embedding(MAX_WORDS, 128, input_length=MAX_LEN))
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test_pad, y_test)
)

# Save model
model.save("toxicity_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Training completed and model saved.")

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1995/1995 ━━━━━━━━━━━━━━━━━━━━ 290s 143ms/step - accuracy: 0.9369 - loss: 0.1970 - val_accuracy: 0.9621 - val_loss: 0.1096
Epoch 2/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 287s 144ms/step - accuracy: 0.9639 - loss: 0.0994 - val_accuracy: 0.9584 - val_loss: 0.1146
Epoch 3/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 284s 142ms/step - accuracy: 0.9686 - loss: 0.0840 - val_accuracy: 0.9621 - val_loss: 0.1179
Epoch 4/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 285s 143ms/step - accuracy: 0.9739 - loss: 0.0695 - val_accuracy: 0.9600 - val_loss: 0.1258
Epoch 5/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 295s 148ms/step - accuracy: 0.9783 - loss: 0.0573 - val_accuracy: 0.9594 - val_loss: 0.1486


Training completed and model saved.


In [ ]:
# Load test data
test_df = pd.read_csv("test.csv", engine="python", on_bad_lines="skip")

# Clean test comments
test_df['comment_text'] = test_df['comment_text'].apply(clean_text)


# Convert to sequences
test_seq = tokenizer.texts_to_sequences(test_df['comment_text'])
test_pad = pad_sequences(test_seq, maxlen=MAX_LEN)

# Predict
predictions = model.predict(test_pad)

# Add predictions to dataframe
test_df['toxicity_score'] = predictions
test_df['toxic_prediction'] = (test_df['toxicity_score'] > 0.5).astype(int)

test_df.head()

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 100s 21ms/step


,id,comment_text,toxicity_score,toxic_prediction
0,00001cee341fdb12,yo bitch ja rule is more succesful then youll ...,0.992824,1
1,0000247867823ef7,from rfc \n\n the title is fine as it is imo,0.000014,0
2,00013b17ad220c46,\n\n sources \n\n zawe ashton on lapland,0.032394,0
3,00017563c3f7919a,if you have a look back at the source the info...,0.000536,0
4,00017695ad8997eb,i dont anonymously edit articles at all,0.000440,0


In [ ]:
test_df['toxic_prediction'] = (test_df['toxicity_score'] > 0.5).astype(int)